# RLHF Lab: Reinforcement Learning from Human Feedback on a Bigram Language Model

## Introduction

In the paper *Training language models to follow instructions with human feedback* (Ouyang et al., 2022), OpenAI introduced **RLHF** — a three-stage pipeline for aligning large language models with human preferences:

1. **Supervised Fine-Tuning (SFT):** Fine-tune a pretrained LLM on high-quality demonstration data.
2. **Reward Model Training:** Train a separate model $R_\phi(x, y)$ to predict which of two completions a human would prefer.
3. **RL Fine-Tuning:** Optimize the SFT model with reinforcement learning (PPO) to maximize the learned reward, while staying close to the SFT model via a KL penalty.

In this lab we build a **miniature, fully functional version** of this pipeline. The key simplifications are:

| | Full-scale RLHF | This Lab |
|---|---|---|
| **Policy** $\pi_\theta$ | Transformer LLM (billions of params) | Bigram (Markov) model (one matrix) |
| **RL algorithm** | PPO (clipped surrogate objective) | REINFORCE (vanilla policy gradient) |
| **Reward signal** | Learned reward model $R_\phi$ | Hand-crafted per-token reward |
| **Reference model** | Frozen copy of SFT model | Frozen copy of the bigram model |
| **KL regularization** | $\beta \cdot \mathrm{KL}(\pi_\theta \| \pi_\mathrm{ref})$ | Same |

Despite these simplifications, every core concept — policy gradients, reward shaping, KL regularization, advantage estimation — transfers directly. The bigram model lets us see the entire mechanism in a few lines of code and train in seconds on a CPU.

## 1. Data Loading and Vocabulary

We begin by loading a corpus of tweets, cleaning the text, and building a vocabulary. These preprocessing utilities live in `utils.py` so we can focus on the RL components here.

In [1]:
import torch
from utils import (
    load_tweets, clean_text, build_vocab, build_count_matrix,
    counts_to_logprobs, BOS, EOS, UNK,
)

CSV_PATH = "trump_tweets_dataset.csv"
VOCAB_SIZE = 10000

# Load and clean
raw_tweets = load_tweets(CSV_PATH)
tokenized = [clean_text(t).split() for t in raw_tweets]
tokenized = [t for t in tokenized if t]
print(f"{len(tokenized):,} non-empty tweets")

# Show a few examples
for i in range(3):
    print(f"  Raw:     {raw_tweets[i][:100]}...")
    print(f"  Cleaned: {' '.join(tokenized[i][:15])}...")
    print()

# Build vocabulary and count matrix
vocab = build_vocab(tokenized, n=VOCAB_SIZE)
counts = build_count_matrix(tokenized, vocab)
print(f"Vocabulary size: {len(vocab)} tokens")

84,559 non-empty tweets
  Raw:     Arrest them all. They are criminals!!! \nForeign ATM? Somali cash exodus from Minneapolis exponentia...
  Cleaned: arrest them all they are criminals nforeign atm somali cash exodus from minneapolis exponentially larger...

  Raw:     When a vehicle is coming at you and is being used as a weapon, deadly force is justified, Nicole Par...
  Cleaned: when a vehicle is coming at you and is being used as a weapon deadly...

  Raw:     It was a Great Honor to speak with the President of Colombia, Gustavo Petro, who called to explain t...
  Cleaned: it was a great honor to speak with the president of colombia gustavo petro who...

Vocabulary size: 10003 tokens


## 2. The Bigram Language Model as a Minimal "LLM"

A bigram model defines a probability distribution over sequences using only one-step-back context:

$$P(w_1, w_2, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{t-1})$$

We store these conditional probabilities in a **transition matrix** $A \in \mathbb{R}^{V \times V}$, where $A_{ij} = P(w_j \mid w_i)$.  In RL language, this matrix *is* our **policy** $\pi_\theta$:

$$\pi_\theta(a_t \mid s_t) = A_{s_t, a_t}$$

where the "state" $s_t = w_{t-1}$ is just the previous token and the "action" $a_t = w_t$ is the next token to emit.

**Analogy to full-scale RLHF:** In a real LLM, the policy $\pi_\theta(a_t \mid s_t)$ is a transformer that conditions on the *entire* preceding context $s_t = (w_1, \ldots, w_{t-1})$. Our bigram model is the extreme simplification where only $w_{t-1}$ matters (the Markov property). Despite this, the RL optimization is structurally identical.

We work in **log-space** for numerical stability: our learnable parameter is $\log A$, and we recover probabilities via softmax when needed. We also freeze a copy as the **reference policy** $\pi_\mathrm{ref}$ — analogous to the frozen SFT model in full RLHF.

In [2]:
# Convert counts to a (V, V) log-probability tensor
logits_ref, key2idx, idx2key = counts_to_logprobs(counts, alpha=1e-3)

V = logits_ref.shape[0]
eos_id = key2idx[EOS]
print(f"Transition matrix shape: {logits_ref.shape}")
print(f"EOS token id: {eos_id}")

# Our learnable policy: starts as a copy of the reference
logits = logits_ref.clone().requires_grad_()

Transition matrix shape: torch.Size([10003, 10003])
EOS token id: 2


## 3. Sampling from the Policy

To generate text, we sample autoregressively:

$$w_t \sim \pi_\theta(\cdot \mid w_{t-1}) = \mathrm{softmax}\bigl(\log A[w_{t-1}, :]\bigr)$$

starting from the special `<bos>` token and continuing until we hit `<eos>` or reach a maximum length.

**Analogy:** This is exactly how LLMs generate text — one token at a time, sampling from the model's distribution conditioned on everything so far.  The only difference is that our "everything so far" is a single token.

The function below samples a batch of $n$ sequences of length $T$ in parallel. We use `torch.no_grad()` because sampling is not a differentiable operation — gradients flow through the *log-probabilities* of the sampled actions, not through the sampling itself (this is the core idea behind REINFORCE).

In [3]:
def sample(logits, n, T):
    """
    Sample n sequences of length T from the policy defined by logits.

    Returns a (n, T) tensor of token indices, starting with <bos>.
    """
    G = torch.empty(n, T, dtype=torch.long)
    G[:, 0] = key2idx[BOS]
    with torch.no_grad():
        for t in range(1, T):
            probs = torch.softmax(logits[G[:, t-1]], dim=1)
            G[:, t] = probs.multinomial(1).squeeze()
    return G


def decode(G, idx2key):
    """Convert a (n, T) tensor of token indices back to readable strings."""
    sentences = []
    for i in range(G.shape[0]):
        words = []
        for t in range(1, G.shape[1]):
            w = idx2key[G[i, t].item()]
            if w == EOS:
                break
            words.append(w)
        sentences.append(" ".join(words))
    return sentences


# Generate a few samples from the (untrained) reference policy
G_demo = sample(logits, 5, 32)
for s in decode(G_demo, idx2key):
    print(s)

rt theleoterrell fined both and regulations promote yourself think your commander in many bills need him
rt erictrump
<unk> to see him well qualified individual taxes champion
https thefederalist com u s time go forward to make america great thanksgiving <unk> hopefully az cont
<unk> for those prayers are fantastic


## 4. Computing Step Log-Probabilities

Given a batch of sampled sequences $G$, we need to compute $\log \pi_\theta(a_t \mid s_t)$ for each step — these are the quantities that REINFORCE differentiates through.

For each timestep $t$, the state is $s_t = G_{:, t-1}$ (the previous token) and the action is $a_t = G_{:, t}$ (the token that was sampled). The log-probability is:

$$\log \pi_\theta(a_t \mid s_t) = \log \mathrm{softmax}(\text{logits}[s_t, :])_{a_t}$$

We also need to handle **end-of-sequence** carefully: once `<eos>` has been emitted, subsequent tokens are padding and should not contribute to the loss or reward. We construct a boolean `valid` mask that is `True` up to and including the first `<eos>`, and `False` thereafter.

**Analogy:** In full-scale RLHF, the same operation happens — you gather the log-probabilities of the tokens the model actually generated. The valid mask corresponds to ignoring padding tokens in a batch of variable-length responses.

In [4]:
def compute_step_log_probs(logits, G):
    """
    Compute per-step log-probabilities and a validity mask.

    Parameters
    ----------
    logits : (V, V) tensor — the log-policy (will be softmax'd)
    G      : (B, T) long tensor — sampled token sequences

    Returns
    -------
    step_logprobs : (B, T-1) — log pi(a_t | s_t) for each step
    action_valid  : (B, T-1) — True where the step is not post-EOS padding
    states        : (B, T-1) — s_t = G[:, :-1]
    actions       : (B, T-1) — a_t = G[:, 1:]
    """
    is_eos = (G == eos_id)
    # A position is valid unless a *previous* position was already EOS
    valid = ~(is_eos.cumsum(dim=1) > 1)

    states = G[:, :-1]
    actions = G[:, 1:]
    action_valid = valid[:, 1:]

    logP = torch.log_softmax(logits, dim=1)
    step_logprobs = logP[states, actions]
    return step_logprobs, action_valid, states, actions

## 5. The Reward Function

In full-scale RLHF, a **learned reward model** $R_\phi(x, y)$ scores complete responses. The reward model is trained from human preference comparisons: given two completions, which does the human prefer?

Here we skip reward model training and define a simple **hand-crafted, per-token reward**:

$$f_t = \begin{cases} -1 + 0.01 & \text{if } a_t \in \text{banned words} \\ 0.01 & \text{otherwise} \end{cases}$$

We choose a set of **banned words** and penalize the model for generating them. The small positive baseline reward (+0.01) encourages the model to keep generating reasonable text rather than collapsing to very short sequences.

This is a simplification in two ways: (1) the reward is per-token rather than per-sequence, and (2) it is hand-crafted rather than learned. But the RL optimization that follows is the same regardless of where the reward comes from.

In [12]:
# Define banned words — the reward penalizes generating these
banned_words = ['obama', 'clinton', 'pelosi', 'kamala']
banned_token_ids = [key2idx[k] for k in banned_words]
banned_mask = torch.zeros(V, dtype=torch.bool)
banned_mask[banned_token_ids] = True

print(f"Banned words: {banned_words}")
print(f"Banned token IDs: {banned_token_ids}")

def compute_step_rewards(actions, action_valid, banned_mask):
    """
    Compute per-step rewards: -1 for banned tokens, 0 otherwise,
    plus a small positive baseline of +0.01.
    """
    is_banned = banned_mask[actions]
    step_rewards = -is_banned.float() + 0.01
    step_rewards = step_rewards.masked_fill(~action_valid, 0.0)
    return step_rewards, is_banned

Banned words: ['the']
Banned token IDs: [3]


## 6. Discounted Returns and Baselines

Raw per-step rewards aren't directly useful for policy gradients — we need the **return**, which measures the total future reward from each timestep. The **discounted return-to-go** at step $t$ is:

$$G_t = \sum_{k=0}^{T-t} \gamma^k \, f_{t+k}$$

where $\gamma \in (0, 1]$ is the discount factor. Discounting down-weights distant rewards, which helps with credit assignment: the model learns that *this particular action* led to reward, not some action 20 steps later.

To reduce variance in the gradient estimate, we subtract a **baseline** and normalize:

$$\hat{A}_t = \frac{G_t - \bar{G}}{\sigma_G}$$

These normalized $\hat{A}_t$ are called **advantages**. Steps with above-average returns get positive advantages (reinforced), and below-average steps get negative advantages (discouraged).

**Analogy to full RLHF:** PPO typically uses a learned **value function** $V_\phi(s_t)$ as the baseline, giving the advantage $\hat{A}_t = G_t - V_\phi(s_t)$. This is more powerful because it's state-dependent, but our simple batch-mean baseline achieves the same variance-reduction goal.

In [13]:
def discounted_return_to_go(step_rewards, action_valid, gamma):
    """
    Compute discounted returns by iterating backwards through the sequence.

    Parameters
    ----------
    step_rewards : (B, T) tensor
    action_valid : (B, T) bool mask
    gamma        : discount factor

    Returns
    -------
    returns : (B, T) tensor of discounted return-to-go values
    """
    step_rewards = step_rewards.masked_fill(~action_valid, 0.0)
    B, T = step_rewards.shape
    returns = torch.zeros_like(step_rewards)
    running = torch.zeros(B, device=step_rewards.device)

    for t in reversed(range(T)):
        running = step_rewards[:, t] + gamma * running
        returns[:, t] = running
        # Reset running return after sequence ends
        running = running * action_valid[:, t]

    return returns

## 7. The REINFORCE Policy Gradient

### Deriving the policy gradient

Our goal is to find parameters $\theta$ that maximize the expected return under our policy:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} R(\tau),$$
where $\tau = (s_1, a_1, s_2, a_2, \ldots)$ is a trajectory sampled from $\pi_\theta$, and $R(\tau)=\sum_{t=1}^T \hat{A}_t$. The challenge is that changing $\theta$ changes the *distribution* over trajectories — we can't just differentiate through the expectation naively.

We can use some algebra (in fact the same trick that leads to the VAE) to resolve this issue.  We write the expectation in integral form (actually a sum for discrete sequences, but whatever).
$$J(\theta) = \int \pi_\theta R(\tau) \mathrm{d} \tau.$$
Taking the gradient and moving it inside the integral (which is allowable here), we have
$$\nabla_\theta J = \int \nabla_\theta (\pi_\theta) R(\tau) + \pi_\theta \nabla_\theta R(\tau) \mathrm{d} \tau.$$
Consider the identity $\nabla_\theta P(\tau) = P(\tau) \nabla_\theta \log P(\tau)$.  This then leads to the following expression  
$$\nabla_\theta J = \int \pi_\theta \nabla_\theta (\log \pi_\theta) R(\tau) + \pi_\theta \nabla_\theta R(\tau) \mathrm{d} \tau.$$  
Writing this in expectation form, we have 
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \nabla_\theta \log \pi_\theta R(\tau) + \nabla_\theta R(\tau)\right].$$

Applying the "future-only" simplification (actions at time $t$ can only affect rewards at times $\geq t$), and making the *assumption* that $\nabla_\theta R(\tau) = 0$ we get the **REINFORCE gradient**
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\!\left[\sum_{t=1}^{T} \hat{A}_t \, \nabla_\theta \log \pi_\theta(a_t \mid s_t)\right]$$

### The surrogate loss

We can't compute this expectation analytically, so we estimate it with a batch of $B$ sampled trajectories. Conveniently, the gradient of the following **surrogate loss** is exactly the (negative) policy gradient:

$$\mathcal{L}(\theta) = -\frac{1}{N_\text{valid}} \sum_{t \in \text{valid}} \hat{A}_t \cdot \log \pi_\theta(a_t \mid s_t)$$

The negative sign converts gradient *ascent* (maximize reward) into gradient *descent* (minimize loss) for use with standard optimizers. Note that $\hat{A}_t$ is treated as a constant (detached from the computation graph) — only $\log \pi_\theta$ carries gradients.

### Why REINFORCE is on-policy, and how PPO relaxes this (if you're interested in PPO vs what we derived above)

A critical feature of the derivation above is that trajectories must be sampled from the **current** policy $\pi_\theta$. After each gradient step, $\theta$ changes, and the old samples are stale — they came from a slightly different distribution. REINFORCE handles this by simply discarding old samples and re-sampling, making it an **on-policy** algorithm.

This is wasteful: we do an expensive round of generation, use those samples for a single gradient step, then throw them away. **PPO** (Proximal Policy Optimization) addresses this by reusing samples across multiple gradient steps. But reusing samples from $\pi_{\theta_\text{old}}$ to estimate the gradient of $\pi_\theta$ requires **importance sampling** — reweighting each sample by the probability ratio:

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)}$$

This gives the off-policy gradient estimate:

$$\nabla_\theta J \approx \mathbb{E}_{\tau \sim \pi_{\theta_\text{old}}}\!\left[\sum_t r_t(\theta) \, \hat{A}_t \, \nabla_\theta \log \pi_\theta(a_t \mid s_t)\right]$$

The problem is that importance ratios can explode. If $\pi_\theta$ assigns high probability to an action that $\pi_{\theta_\text{old}}$ rarely sampled, $r_t(\theta)$ becomes very large, producing enormous, noisy gradient updates. PPO's predecessor, **TRPO** (Trust Region Policy Optimization), addressed this by constraining the KL divergence between $\pi_\theta$ and $\pi_{\theta_\text{old}}$ at each update — effectively a hard trust region. PPO replaces this with a simpler **clipped surrogate**:

$$\mathcal{L}^{\text{PPO}} = -\mathbb{E}\!\left[\min\!\Big(r_t(\theta)\,\hat{A}_t,\;\text{clip}(r_t(\theta),\, 1{-}\epsilon,\, 1{+}\epsilon)\,\hat{A}_t\Big)\right]$$

The clipping caps the importance ratio at $1 \pm \epsilon$, preventing any single sample from producing a disproportionately large update. This is the price of going off-policy: you gain sample efficiency (reuse old trajectories for multiple updates) but must guard against the instability of importance weights.

**In this lab** we use on-policy REINFORCE, which avoids this complexity entirely — each batch of samples is fresh from the current policy, so no importance correction is needed. The tradeoff is that we need more samples overall, but for our tiny bigram model this is perfectly fine.  GPT-3's RLHF scheme uses PPO because it is too expensive to draw a batch of generations at every step - instead they keep a given batch of samples frozen for a (hyperparemeter) number of steps, using PPO instead of policy gradients to account for these samples being stale relative to the current policy, and then recomputing samples.  

## 8. KL Regularization

Without any constraint, the RL optimizer would happily destroy the language model's coherence in pursuit of reward — it might learn to output only a handful of "safe" tokens repeatedly. This is called **reward hacking** or **mode collapse**.

The solution, and one of the key contributions of the InstructGPT paper, is to add a **KL penalty** that keeps the fine-tuned policy close to a reference (pre-RL) policy:

$$f_t^{\text{shaped}} = f_t - \beta \cdot \underbrace{\left[\log \pi_\theta(a_t \mid s_t) - \log \pi_\text{ref}(a_t \mid s_t)\right]}_{\text{per-step KL divergence estimate}}$$

The parameter $\beta$ controls the strength of regularization:
- **Large $\beta$**: the model barely changes from the reference — safe but limited alignment.
- **Small $\beta$**: the model is free to change more — better reward optimization but risk of degeneration.

This per-step KL term $\log \pi_\theta(a_t|s_t) - \log \pi_\text{ref}(a_t|s_t)$ is an unbiased, single-sample estimate of the KL divergence $D_\text{KL}(\pi_\theta \| \pi_\text{ref})$ at state $s_t$.

**Analogy:** This is *identical* to the KL penalty in full-scale RLHF. The InstructGPT paper penalizes $\beta \cdot D_\text{KL}(\pi_\theta \| \pi_\text{SFT})$ for exactly the same reason — to prevent the model from drifting too far from the supervised fine-tuned starting point.

## 9. Training Loop

We now assemble all the pieces into a training loop. Each iteration:

1. **Sample** a batch of sequences from the current policy $\pi_\theta$
2. **Score** them: compute log-probs under both $\pi_\theta$ and $\pi_\text{ref}$
3. **Reward**: compute per-step rewards and KL penalties
4. **Returns**: compute discounted return-to-go with KL-shaped rewards
5. **Advantages**: normalize returns to get $\hat{A}_t$
6. **Update**: compute the REINFORCE loss and backpropagate

In [14]:
# Hyperparameters
beta = 0.1          # KL penalty strength
gamma = 0.97        # discount factor
lr = 0.1            # learning rate
n_iters = 100       # training iterations
batch_size = 512    # sequences per batch
seq_len = 64        # max sequence length

# Reset the learnable policy to the reference
logits = logits_ref.clone().requires_grad_()
optimizer = torch.optim.Adam([logits], lr=lr)

print(f"{'iter':>4s}  {'loss':>8s}  {'banned%':>8s}  {'reward':>8s}  {'KL':>8s}")
print("-" * 44)

for j in range(n_iters):
    # 1. Sample from current policy
    G = sample(logits, batch_size, seq_len)

    # 2. Compute log-probs under current and reference policies
    step_log_probs, action_valid, states, actions = compute_step_log_probs(logits, G)
    ref_log_probs, _, _, _ = compute_step_log_probs(logits_ref, G)

    # 3. Per-step rewards
    step_rewards, is_banned = compute_step_rewards(actions, action_valid, banned_mask)

    # 4. KL penalty: per-step estimate of KL(pi_theta || pi_ref)
    kl_step = step_log_probs - ref_log_probs
    kl_step = kl_step.masked_fill(~action_valid, 0.0)

    # 5. Shape rewards with KL penalty
    shaped_rewards = step_rewards - beta * kl_step
    shaped_rewards = shaped_rewards.masked_fill(~action_valid, 0.0)

    # 6. Discounted returns-to-go (using KL-shaped rewards)
    returns = discounted_return_to_go(shaped_rewards, action_valid, gamma=gamma)

    # 7. Advantage estimation: normalize over valid steps
    baseline = returns[action_valid].mean()
    advantages = returns - baseline
    advantages = advantages.masked_fill(~action_valid, 0.0)

    adv_valid = advantages[action_valid]
    advantages = (advantages - adv_valid.mean()) / (adv_valid.std() + 1e-8)
    advantages = advantages.masked_fill(~action_valid, 0.0)

    # 8. REINFORCE loss
    loss = -(advantages.detach() * step_log_probs)
    loss = loss.masked_fill(~action_valid, 0.0)
    loss = loss.sum() / action_valid.sum()

    # 9. Gradient step
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Logging
    banned_rate = is_banned[action_valid].float().mean().item()
    mean_kl = kl_step[action_valid].mean().item()
    mean_reward = step_rewards[action_valid].mean().item()

    if j % 1 == 0 or j == n_iters - 1:
        print(f"{j:4d}  {loss.item():8.4f}  {100*banned_rate:7.4f}%  {mean_reward:8.4f}  {mean_kl:8.4f}")

iter      loss   banned%    reward        KL
--------------------------------------------
   0    0.1070   3.7217%   -0.0272    0.0000
   1    0.0692   3.4579%   -0.0246    0.0033
   2    0.0803   3.0117%   -0.0201    0.0076
   3    0.0794   2.7267%   -0.0173    0.0096
   4    0.0481   2.5570%   -0.0156    0.0196
   5    0.0577   2.4031%   -0.0140    0.0235
   6    0.0178   2.1026%   -0.0110    0.0357
   7    0.0148   1.9871%   -0.0099    0.0412
   8    0.0124   1.8746%   -0.0087    0.0470
   9    0.0188   1.9807%   -0.0098    0.0491
  10    0.0170   1.3176%   -0.0032    0.0675
  11    0.0751   1.3903%   -0.0039    0.0800
  12    0.0053   1.4005%   -0.0040    0.0895
  13    0.0265   1.3349%   -0.0033    0.0911
  14    0.1033   1.2781%   -0.0028    0.0956
  15    0.0426   1.0682%   -0.0007    0.1055
  16    0.0581   1.1456%   -0.0015    0.1138
  17    0.1734   0.9220%    0.0008    0.1156
  18    0.0363   0.8750%    0.0012    0.1332
  19    0.0022   0.8843%    0.0012    0.1306
  20    0.

KeyboardInterrupt: 

## 10. Results and Exploration

Let's compare samples from the **reference** (pre-RL) policy and the **fine-tuned** policy to see the effect of training.

In [15]:
print("=== Reference policy (before RL) ===")
G_ref = sample(logits_ref, 10, 32)
for s in decode(G_ref, idx2key):
    print(f"  {s}")

print()
print("=== Fine-tuned policy (after RL) ===")
G_tuned = sample(logits, 10, 32)
for s in decode(G_tuned, idx2key):
    print(f"  {s}")

# Check: how often do banned words appear?
_, valid_ref, _, actions_ref = compute_step_log_probs(logits_ref, G_ref)
_, valid_tuned, _, actions_tuned = compute_step_log_probs(logits, G_tuned)
ref_banned = banned_mask[actions_ref][valid_ref].float().mean().item()
tuned_banned = banned_mask[actions_tuned][valid_tuned].float().mean().item()
print(f"\nBanned word rate — reference: {ref_banned:.4f}, fine-tuned: {tuned_banned:.4f}")

=== Reference policy (before RL) ===
  my past time left democrat az house but i am now will deliver remarks on
  rt <unk> <unk> sen mitt
  and failure at trump tower to ad html unsuccessfully lesley crystal clear that owner call me twice as of the presidency will bring star celebrity apprentice because they are a york
  we re cowards that and blackmail hosted by the white house lawyer alina notice the fake witch hunt in their <unk> s great again making their way to put crooked hillary
  poll lead the shutdown that china e <unk> com politics where start
  rt realamericasvoice and all of a chance to believe that zero chance of america great big names many auto and the second amendment freedom
  rt realdonaldtrump him
  a national committee s a fine is in north carolina n <unk> will be signing the late we always supports joe biden is proud reminder actforamerica imagine the years as <unk>
  rt no other assets in fact that the area and go anywhere near lyin acceptable while i think like p

## Exercises

1. **Change the banned words.** Try banning different sets of words (e.g., common words like "the" or "is"). How does the model adapt? Does it remain coherent?

2. **Vary $\beta$.** Try $\beta = 0$ (no KL penalty), $\beta = 0.1$, $\beta = 10$. What happens to sample quality and banned-word avoidance in each case?

3. **Reward shaping.** The current loop uses KL-shaped rewards in the return computation. Try using *only* the raw task reward (remove the `- beta * kl_step` from `shaped_rewards`). Does the model still converge? What happens to the KL divergence?

4. **Discount factor.** Try $\gamma = 1.0$ (undiscounted) vs. $\gamma = 0.5$ (heavy discounting). How does this affect credit assignment?

5. **Batch size.** Try `batch_size = 64` vs. `batch_size = 4096`. How does this affect training stability?